## Vision Agents with SmolAgents

In [1]:
%pip install smolagents

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from huggingface_hub import notebook_login

notebook_login()

### Providing Images at the Start of the Agent's Execution

In this approach, images are passed to the agent at the start and stored as task_images alongside the task prompt. The agent then processes these images throughout its execution.

Consider the case where Alfred wants to verify the identities of the superheroes attending the party. He already has a dataset of images from previous parties with the names of the guests. Given a new visitor's image, the agent can compare it with the existing dataset and make a decision about letting them in.

In this case, a guest is trying to enter, and Alfred suspects that this visitor might be The Joker impersonating Wonder Woman. Alfred needs to verify their identity to prevent anyone unwanted from entering.

Let’s build the example. First, the images are loaded. In this case, we use images from Wikipedia to keep the example minimal, but imagine the possible use-cases!

In [3]:
from PIL import Image
import requests
from io import BytesIO

image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg",
    "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg"
]

images = []
for url in image_urls:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36" 
    }
    response = requests.get(url,headers=headers)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    images.append(image)

Now that we have the images, the agent will tell us wether the guests is actually a superhero (Wonder Woman) or a villian (The Joker).

In [ ]:
import os
from dotenv import load_dotenv
from smolagents import CodeAgent, OpenAIServerModel, InferenceClientModel

load_dotenv()  # reads HF_TOKEN from the .env file next to this notebook

hf_token = os.environ.get("HF_TOKEN")

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-VL-72B-Instruct", #This model is designed for vision-language tasks, making it suitable for analyzing images and generating descriptions based on visual input.
    token=hf_token
)

# Instantiate the agent
agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=20,
    verbosity_level=2
)

response = agent.run(
    """
    Describe the costume and makeup that the comic character in these photos is wearing and return the description.
    Tell me if the guest is The Joker or Wonder Woman.
    """,
    images=images
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Describe the costume and makeup that the comic character in these photos is wearing and return the description. │
│     Tell me if the guest is The Joker or Wonder Woman.                                                          │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-VL-72B-Instruct ───────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To describe the costume and makeup of the character in the images, I'll analyze the visual elements such  
as clothing, colors, and makeup style. Then, I'll determine whether the character is The Joker or Wonder Woman     
based on those observations.                                                                                       
                                                                                                                   
Code:                                                                                                              
<code>                                                                                                             
import re                                                                                                          
                                                                                                                   
# Analyzing the first image                                                                                        
costume_description_1 = "The character is wearing a purple suit jacket over a yellow shirt with a purple bow tie.  
The makeup includes white face paint, exaggerated red lips forming a wide grin, and dark eye makeup."              
character_guess_1 = "The Joker"                                                                                    
                                                                                                                   
# Analyzing the second image                                                                                       
costume_description_2 = "The character is dressed in a black suit with a white shirt underneath. The makeup        
features white face paint, green hair, and a wide red grin. There is also a flower pinned to the suit."            
character_guess_2 = "The Joker"                                                                                    
                                                                                                                   
# Combining descriptions and guesses                                                                               
combined_description = f"In the first photo, the character is described as follows: {costume_description_1}. This  
suggests the character is {character_guess_1}. In the second photo, the character is described as follows:         
{costume_description_2}. This also suggests the character is {character_guess_2}."                                 
                                                                                                                   
print(combined_description)                                                                                        
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import re                                                                                                        
                                                                                                                   
  # Analyzing the first image                                                                                      
  costume_description_1 = "The character is wearing a purple suit jacket over a yellow shirt with a purple bow     
  tie. The makeup includes white face paint, exaggerated red lips forming a wide grin, and dark eye makeup."       
  character_guess_1 = "The Joker"                                                                                  
                                                                                                                   
  # Analyzing the second image                                                                                     
  costume_description_2 = "The character is dressed in a black suit with a white shirt underneath. The makeup      
  features white face paint, green hair, and a wide red grin. There is also a flower pinned to the suit."          
  character_guess_2 = "The Joker"                                                                                  
                                                                                                                   
  # Combining descriptions and guesses                                                                             
  combined_description = f"In the first photo, the character is described as follows: {costume_description_1}.     
  This suggests the character is {character_guess_1}. In the second photo, the character is described as follows:  
  {costume_description_2}. This also suggests the character is {character_guess_2}."                               
                                                                                                                   
  print(combined_description)                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
In the first photo, the character is described as follows: The character is wearing a purple suit jacket over a 
yellow shirt with a purple bow tie. The makeup includes white face paint, exaggerated red lips forming a wide grin,
and dark eye makeup.. This suggests the character is The Joker. In the second photo, the character is described as 
follows: The character is dressed in a black suit with a white shirt underneath. The makeup features white face 
paint, green hair, and a wide red grin. There is also a flower pinned to the suit.. This also suggests the 
character is The Joker.

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: Based on the analysis of the two images, the character depicted is clearly The Joker. Both images show the
iconic makeup and costume associated with the character, including the white face paint, exaggerated red lips, and 
distinctive clothing.                                                                                              
                                                                                                                   
Code:                                                                                                              
<code>                                                                                                             
final_answer("The character in both photos is The Joker. The costume consists of a purple suit jacket over a yellow
shirt with a purple bow tie in the first photo, and a black suit with a white shirt underneath in the second photo.
The makeup includes white face paint, exaggerated red lips forming a wide grin, and green hair in the second       
photo.")                                                                                                           
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The character in both photos is The Joker. The costume consists of a purple suit jacket over a     
  yellow shirt with a purple bow tie in the first photo, and a black suit with a white shirt underneath in the     
  second photo. The makeup includes white face paint, exaggerated red lips forming a wide grin, and green hair in  
  the second photo.")                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The character in both photos is The Joker. The costume consists of a purple suit jacket over a yellow
shirt with a purple bow tie in the first photo, and a black suit with a white shirt underneath in the second photo.
The makeup includes white face paint, exaggerated red lips forming a wide grin, and green hair in the second photo.

[Step 2: Duration 6.27 seconds| Input tokens: 7,151 | Output tokens: 389]